# Ajuste de Hiperparâmetros

Este módulo funciona como um laboratório de testes antes da execução da esteira principal de produção. O objetivo estrutural desta etapa é remover as suposições fixas (como definir arbitrariamente 3 grupos no K-Means ou 1% de fraudes no Isolation Forest) e permitir que os dados justifiquem essas escolhas matematicamente.

Método do Cotovelo e Silhueta para K-Means: Testaremos múltiplas configurações de grupos (de 2 a 8 clusters) e avaliaremos o equilíbrio perfeito entre a compactação interna de cada grupo (Inércia) e a separação clara entre eles (Score de Silhueta). Isso provará estatisticamente qual é o número ideal de perfis de consumo.

Mapeamento Contínuo do Isolation Forest: Em vez de forçar o algoritmo a isolar uma taxa fixa de anomalias (contamination), vamos extrair a pontuação bruta de risco (Escore de Anomalia) de toda a base. Através da distribuição desses escores em um histograma, poderemos visualizar com clareza onde a cauda de comportamentos atípicos realmente começa, definindo um limiar de corte baseado na realidade dos dados.

# 1. Validação Matemática do K-Means (Escolha do 'K' Ideal)

Importação e Execução do Teste de Clusters

In [1]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns

print("Iniciando bateria iterativa de testes do K-Means...")

# 1. Selecionando as variáveis normalizadas de comportamento geral
X_kmeans = df[['VOLUME_TOTAL_LOG_SCALED', 'TICKET_MEDIO_LOG_SCALED']].values

inercia = []
silhuetas = []
k_range = range(2, 9)

# 2. Treinando o modelo repetidas vezes para cada valor de K
for k in k_range:
    kmeans_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans_test.fit_predict(X_kmeans)
    inercia.append(kmeans_test.inertia_)
    silhuetas.append(silhouette_score(X_kmeans, labels))

# 3. Construção do Gráfico de Eixo Duplo para Tomada de Decisão
fig, ax1 = plt.subplots(figsize=(10, 6))

# Eixo Y esquerdo (Inércia / Compactação)
cor_inercia = '#e74c3c'
ax1.set_xlabel('Número de Clusters (k)', fontsize=12)
ax1.set_ylabel('Inércia (Compactação interna)', color=cor_inercia, fontsize=12)
ax1.plot(k_range, inercia, marker='o', color=cor_inercia, linewidth=2, label='Inércia')
ax1.tick_params(axis='y', labelcolor=cor_inercia)

# Eixo Y direito (Silhueta / Separação)
ax2 = ax1.twinx()  
cor_silhueta = '#2980b9'
ax2.set_ylabel('Score de Silhueta (Qualidade da separação)', color=cor_silhueta, fontsize=12)
ax2.plot(k_range, silhuetas, marker='s', color=cor_silhueta, linewidth=2, label='Silhueta')
ax2.tick_params(axis='y', labelcolor=cor_silhueta)

plt.title('Validação de Hiperparâmetros: Método do Cotovelo vs Silhueta', fontsize=14)
fig.tight_layout()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

Iniciando bateria iterativa de testes do K-Means...


NameError: name 'df' is not defined

# 2. Dimensionamento Real do Isolation Forest (Limiar de Anomalia)

Mapeamento de Distribuição e Identificação do Ponto de Corte

In [2]:
from sklearn.ensemble import IsolationForest

print("Iniciando análise contínua de distribuição do Isolation Forest...")

# 1. Instanciando o modelo focando em estabilidade (n_estimators=200) e sem contaminação fixa
modelo_if_lab = IsolationForest(
    n_estimators=200, 
    contamination='auto', 
    max_samples='auto', 
    random_state=42
)

# 2. Treinando e extraindo o Escore de Anomalia Bruto (Decision Function)
# Obs: Valores menores/negativos são MAIS anômalos.
modelo_if_lab.fit(df[features_modelos])
df['anomaly_score'] = modelo_if_lab.decision_function(df[features_modelos])

# 3. Calculando percentis estatísticos para demarcar os limites de risco no gráfico
p_01 = df['anomaly_score'].quantile(0.01)
p_05 = df['anomaly_score'].quantile(0.05)

# 4. Construção do Histograma de Frequência dos Escores
plt.figure(figsize=(12, 6))
sns.histplot(df['anomaly_score'], bins=50, kde=True, color='#8e44ad', alpha=0.6)

# Linhas demarcatórias para apoiar a decisão de negócio
plt.axvline(x=p_01, color='#e74c3c', linestyle='--', linewidth=2, label=f'Top 1% mais anômalo (Score <= {p_01:.3f})')
plt.axvline(x=p_05, color='#e67e22', linestyle='--', linewidth=2, label=f'Top 5% mais anômalo (Score <= {p_05:.3f})')
plt.axvline(x=0, color='#27ae60', linestyle='-', linewidth=2, label='Limiar Padrão (Normal > 0 > Anômalo)')

plt.title('Distribuição Contínua do Escore de Anomalia (Isolation Forest)', fontsize=14)
plt.xlabel('Escore de Anomalia (Valores negativos indicam maior risco)', fontsize=12)
plt.ylabel('Volume de Portadores (Frequência)', fontsize=12)
plt.legend(loc='upper left', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

Iniciando análise contínua de distribuição do Isolation Forest...


NameError: name 'df' is not defined